# 02 · Construir la serie mensual con cobertura y media móvil

Práctica basada en `notebooks/02_energia.ipynb`. Tiempo sugerido: 20–30 minutos.

**Tu tarea:** completá únicamente la función de la sección 3.
La carga de datos, el entorno y las pruebas ya están preparados.

1. Elegí el kernel del entorno `.venv` del proyecto y ejecutá las celdas de preparación.
2. Leé el contrato y reemplazá el `raise NotImplementedError` por tu implementación.
3. Ejecutá de nuevo la celda de la función y después la de pruebas.
4. Si todas pasan, esa misma celda probará tu función sobre los datos reales y mostrará el resultado.

Podés usar **Ejecutar todo** desde un kernel nuevo. Mientras la función esté pendiente,
la última celda mostrará un aviso; no tenés que instalar nada adicional en el entorno actual.

## 1. Preparación lista

La base equivale a `base` del notebook original: fecha como índice, nombres preparados y faltantes, negativos y ceros conservados.

Los archivos de `../datos/` ya están descargados. Si faltan, se descargan una vez desde Kaggle y se guardan localmente.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from pandas.testing import assert_frame_equal

# Encuentra la carpeta tanto desde la raíz del proyecto como desde este notebook.
ubicaciones = [Path.cwd(), *Path.cwd().parents]
carpeta = next(
    (
        candidata
        for ubicacion in ubicaciones
        for candidata in [ubicacion, ubicacion / "practicas_livecoding"]
        if candidata.name == "practicas_livecoding"
        and (candidata / "preparacion.py").is_file()
    ),
    None,
)
if carpeta is None:
    raise FileNotFoundError("Abrí el notebook dentro de la carpeta practicas_livecoding.")

if str(carpeta) not in sys.path:
    sys.path.insert(0, str(carpeta))

from preparacion import cargar_viviendas, cargar_energia

pd.set_option("display.max_columns", 20)

In [2]:
datos = cargar_energia()

print(f"Filas: {len(datos):,} | Columnas: {len(datos.columns)}")
display(datos.head())

Filas: 285,499 | Columnas: 13


## 2. Consigna y resultado esperado

A partir de la base completa con índice `Data`, construí una tabla mensual con
frecuencia de inicio de mes (`MS`) e índice llamado `Data`. Incluí todos los meses
entre la primera y la última fecha, aunque alguno no tenga registros.

| Columna, en este orden | Cálculo |
|---|---|
| `consumo_total` | Suma de `Consumo` con `min_count=1` |
| `consumidores_total` | Suma de `Consumidores` con `min_count=1` |
| `registros` | Cantidad de filas del mes, incluidos los faltantes |
| `consumo_presente` | Cantidad de valores de `Consumo` no nulos |
| `cobertura_pct` | `consumo_presente / registros * 100` |
| `media_movil_12` | Media de `consumo_total` en los últimos 12 meses, incluido el actual |

La media móvil exige 12 valores mensuales válidos: usá `min_periods=12`, sin centrar.
Un mes sin mediciones no se convierte en cero. Un mes sin filas tiene ambos conteos en cero
y cobertura `NaN`. Conservá los negativos. No modifiques la entrada; podés asumir que no está vacía.

## 3. Tu función

Editá solo esta celda. Mantené el nombre de la función y sus parámetros para que funcionen las pruebas.

In [3]:
def resumir_consumo_mensual(datos):
    """Implementá el contrato de la sección 2 sin modificar datos."""
    # TODO: escribí tu solución acá.
    raise NotImplementedError("Completá esta función y volvé a ejecutar las pruebas.")

## 4. Pruebas y ejecución real

Un `AssertionError` indica qué resultado difiere del esperado. Primero revisá el caso pequeño que falló. Después de editar la función, ejecutá su celda antes de volver a correr estas pruebas.

Las pruebas verifican ejemplos pequeños y propiedades del resultado real; no sustituyen una revisión de tu razonamiento.

In [4]:
# Estas pruebas no reemplazan tu función: comprueban su resultado.
def comparar(actual, esperado):
    assert isinstance(actual, pd.DataFrame), "La función debe devolver un DataFrame."
    assert_frame_equal(
        actual, esperado,
        check_dtype=False, check_index_type=False, check_column_type=False,
        check_freq=False, rtol=1e-6, atol=1e-8,
    )


def probar(funcion):
    fechas = pd.date_range("2023-01-01", periods=14, freq="MS", name="Data")
    caso = pd.DataFrame({
        "Consumo": [10, np.nan, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140],
        "Consumidores": [1, np.nan, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],
    }, index=fechas)
    extra = pd.DataFrame(
        {"Consumo": [5], "Consumidores": [1]},
        index=pd.DatetimeIndex(["2023-01-01"], name="Data"),
    )
    caso = pd.concat([caso, extra])
    original = caso.copy(deep=True)
    esperado = pd.DataFrame({
        "consumo_total": [15, np.nan, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140],
        "consumidores_total": [2, np.nan, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],
        "registros": [2] + [1] * 13,
        "consumo_presente": [2, 0] + [1] * 12,
        "cobertura_pct": [100, 0] + [100] * 12,
        "media_movil_12": [np.nan] * 13 + [85],
    }, index=fechas)
    comparar(funcion(caso), esperado)

    # Febrero no tiene filas; enero tiene un ajuste negativo.
    con_hueco = caso.iloc[[0, 2]].copy()
    con_hueco.iloc[0, 0] = -10
    esperado_hueco = pd.DataFrame({
        "consumo_total": [-10, np.nan, 30],
        "consumidores_total": [1, np.nan, 3],
        "registros": [1, 0, 1],
        "consumo_presente": [1, 0, 1],
        "cobertura_pct": [100, np.nan, 100],
        "media_movil_12": [np.nan] * 3,
    }, index=fechas[:3])
    comparar(funcion(con_hueco), esperado_hueco)
    comparar(caso, original)
    print("OK: meses repetidos, mes ausente, faltantes, negativos y ventana de 12 meses.")


resultado = None
try:
    probar(resumir_consumo_mensual)
except NotImplementedError:
    print("PENDIENTE: completá la función de la sección 3 y volvé a ejecutar esta celda.")
else:
    # Comprobamos también que el dataset real permanezca intacto.
    antes = datos.copy(deep=True)
    resultado = resumir_consumo_mensual(datos)
    comparar(datos, antes)
    assert resultado.index.is_monotonic_increasing and resultado.index.is_unique
    assert resultado["registros"].sum() == len(datos)
    assert resultado["consumo_presente"].sum() == datos["Consumo"].notna().sum()
    np.testing.assert_allclose(resultado["consumo_total"].sum(), datos["Consumo"].sum())
    np.testing.assert_allclose(resultado["consumidores_total"].sum(), datos["Consumidores"].sum())
    assert resultado["media_movil_12"].iloc[:11].isna().all()
    print(f"OK: dataset real. Resultado: {len(resultado):,} filas.")
    display(resultado.head(20))

PENDIENTE: completá la función de la sección 3 y volvé a ejecutar esta celda.


## Pista opcional

<details>
<summary>Abrir solo si te trabás</summary>

`resample('MS')`, `sum(min_count=1)`, diferencia entre `size` y `count`, y `rolling`.

</details>

**Para explicar en voz alta:** ¿qué filas entran al cálculo?, ¿qué representa cada fila de la salida?, ¿qué caso límite cubre cada prueba?